## Analysis of a single simulation

In [ ]:
import os
import pandas as pd
import numpy as np
from src.utils import load_data, compute_fourier, geom_rank_cell_nodes, get_firingrate_hist, get_plv, extract_main_cell_types
from src.plotting import plot_fourier_analysis, psth_plot, multiple_psth_plot

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
%matplotlib tk

In [ ]:
import matplotlib
matplotlib.rcParams.update({'font.size': 12})

In [ ]:
config_path = '/mnt/e/tACS/fromOBI/hex0_10Hz_1Vm/sim_with_stim.json'

In [ ]:
import bluepysnap as bp

In [ ]:
s = bp.Simulation(config_path)

In [ ]:
simulationData, allData, cellData, spikeData = load_data(config_path)

In [ ]:
spikeData

In [ ]:
cellData = s.circuit.nodes['S1nonbarrel_neurons'].get(spikeData.values)

In [ ]:
s.config['node_set']

In [ ]:
cell_to_rank = geom_rank_cell_nodes(cellData, simulationData)

In [ ]:
plt.figure()
plt.hist(spikeData.index,bins=int(2500/5))
plt.show()

In [ ]:
from run import *
hist = get_smoothed_firing_rate(
        simulationData,
        spikeData,
        n_neurons=31090,
        delay=1500,
        time_stop=2500,
        bin_width=5,
    )

In [ ]:
xf_pos_HC, amplitude_HC, yf_HC = compute_fourier(hist, 5 * 1e-3)

In [ ]:
plt.figure()
plt.stem(xf_pos_HC[1:], amplitude_HC[1:], linefmt='C0-', markerfmt='C0o', basefmt=' ')
plt.stem(xf_pos_HF[1:], amplitude_HF[1:], linefmt='C1-', markerfmt='C1o', basefmt=' ')
plt.legend(['10 Hz TI','HF control'])
plt.ylabel('Normalized Fourier Amplitude')
plt.xlabel('Frequency')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure()
t = 0.005*np.arange(len(hist))
plt.plot(t, hist)
plt.plot(t,0.2+0.05*np.sin(2*np.pi*10*t))
plt.show()

## simulations

In [ ]:
#Choose a simulation you want to look at
sim_id = '10Hz4Vm_delay25'
#choose bin size for firing rate calculation
bin = 5
#get the location of the config file for that simulation
config_path = simulations.loc[simulations['sim_id'] == sim_id, 'source_file'].values[0]
freq = simulations.loc[simulations['sim_id'] == sim_id, 'stim_freq'].values[0]
delay = simulations.loc[simulations['sim_id'] == sim_id, 'analysis_start'].values[0]
time_stop = simulations.loc[simulations['sim_id'] == sim_id, 'analysis_stop'].values[0]
#load the data for that simulation
print(f"Loading data for simulation {sim_id} from {config_path}")
output_folder = os.path.join("output", sim_id)
os.makedirs(output_folder, exist_ok=True)

In [ ]:
delay

In [ ]:

simulationData, allData, cellData, spikeData = load_data(config_path)
cell_to_rank = geom_rank_cell_nodes(cellData, simulationData)

# cell_types = extract_main_cell_types(spikeData, cellData, cell_to_rank)
# spikeData, cell_to_rank = cell_types['INT_INH']

psth_plot(spikeData, "all cells", cell_to_rank, simulationData, bin_width=bin, save_histogram=os.path.join(output_folder, f"psth_{sim_id}_{bin}bin_smoothed10.png"), smoothed=True,figsize=(12,7))


In [ ]:
from run import *
hist = get_smoothed_firing_rate(
        simulationData,
        spikeData,
        n_neurons=31090,
        delay=delay,
        time_stop=time_stop,
        bin_width=bin,
    )

In [ ]:
hist

In [ ]:
plt.figure()
plt.plot(hist)

In [ ]:
delay = 1832.5 - 1825
print(delay *1e-3 * 360/0.1)

In [ ]:
delay = 1862.5 - 1850
print(delay *1e-3 * 360/0.1)

In [ ]:
delay = 2057.5 - 2050
print(delay *1e-3 * 360*7)

In [ ]:
delay = 1825-1807
print(delay *1e-3 * 360*7)

In [ ]:
xf_pos, amplitude, yf = compute_fourier(hist, bin * 1e-3)
#yf = yf * np.exp(-1j * (np.pi/2 +np.pi))

In [ ]:
np.angle(yf[10])*180/np.pi

In [ ]:
#Choose a simulation you want to look at
sim_id = '10Hz4Vm_delay25'
#choose bin size for firing rate calculation
bin = 5
#get the location of the config file for that simulation
config_path = simulations.loc[simulations['sim_id'] == sim_id, 'source_file'].values[0]
freq = simulations.loc[simulations['sim_id'] == sim_id, 'stim_freq'].values[0]
delay = simulations.loc[simulations['sim_id'] == sim_id, 'analysis_start'].values[0]
time_stop = simulations.loc[simulations['sim_id'] == sim_id, 'analysis_stop'].values[0]

simulationData_lc, allData_lc, cellData_lc, spikeData_lc = load_data(config_path)
cell_to_rank_lc = geom_rank_cell_nodes(cellData_lc, simulationData_lc)

In [ ]:
simulationData_lc, allData_lc, cellData_lc, spikeData_lc = load_data(config_path)
cell_to_rank_lc = geom_rank_cell_nodes(cellData_lc, simulationData_lc)

# cell_types_lc = extract_main_cell_types(spikeData_lc, cellData_lc, cell_to_rank_lc)
# spikeData_lc, cell_to_rank_lc = cell_types_lc['INT_INH']

In [ ]:
multiple_psth_plot([spikeData_lc,spikeData], "", [cell_to_rank_lc,cell_to_rank], [simulationData_lc,simulationData],['highcorrelation baseline','baseline'], bin_width=bin, save_histogram=os.path.join(output_folder, f"psth_{sim_id}_{bin}bin_smoothed10.png"), figsize=(6,3),colorList = ['C1','k'], normalized=True)


In [ ]:
hist, _ = get_firingrate_hist(
        simulationData_lc, spikeData_lc, bin_width=bin,
        n_neurons=cell_to_rank_lc.shape[0], delay=delay , time_stop=time_stop, smoothed=True
    )
xf_pos, amplitude, yf = compute_fourier(hist, bin * 1e-3, referenced=True, sim_id=sim_id)
phase = np.angle(yf)

In [ ]:
hist_baseline, _ = get_firingrate_hist(
        simulationData_lc, spikeData_lc, bin_width=bin,
        n_neurons=cell_to_rank_lc.shape[0], delay=500 , time_stop=1500, smoothed=True
    )
_, amplitude_baseline, _ = compute_fourier(hist_baseline, bin * 1e-3, referenced=True, sim_id=sim_id)


In [ ]:
#Choose a simulation you want to look at
sim_id = '10Hz1Vm_highCorrelation'
#choose bin size for firing rate calculation
bin = 5
#get the location of the config file for that simulation
config_path = simulations.loc[simulations['sim_id'] == sim_id, 'source_file'].values[0]
freq = simulations.loc[simulations['sim_id'] == sim_id, 'stim_freq'].values[0]
delay = simulations.loc[simulations['sim_id'] == sim_id, 'analysis_start'].values[0]
time_stop = simulations.loc[simulations['sim_id'] == sim_id, 'analysis_stop'].values[0]
#load the data for that simulation
print(f"Loading data for simulation {sim_id} from {config_path}")
output_folder = os.path.join("output", sim_id)
os.makedirs(output_folder, exist_ok=True)

In [ ]:
simulationData_hc, allData_hc, cellData_hc, spikeData_hc = load_data(config_path)
cell_to_rank_hc = geom_rank_cell_nodes(cellData_hc, simulationData_hc)

hist_hc, _ = get_firingrate_hist(
        simulationData_hc, spikeData_hc, bin_width=bin,
        n_neurons=cell_to_rank_hc.shape[0], delay=delay , time_stop=time_stop, smoothed=True
    )
xf_pos_hc, amplitude_hc, yf_hc = compute_fourier(hist_hc, bin * 1e-3, referenced=True, sim_id=sim_id)
phase_hc = np.angle(yf_hc)

In [ ]:
hist_baseline_hc, _ = get_firingrate_hist(
        simulationData_hc, spikeData_hc, bin_width=bin,
        n_neurons=cell_to_rank_hc.shape[0], delay=500 , time_stop=1500, smoothed=True
    )
_, amplitude_baseline_hc, _ = compute_fourier(hist_baseline_hc, bin * 1e-3, referenced=True, sim_id=sim_id)


In [ ]:
plot_fourier_analysis(
            xf_pos, [amplitude,amplitude_hc], [phase,phase_hc], freq,
            save_path=os.path.join(output_folder, f"fourier_baselineNormalized_{bin}bin_{sim_id}.png"),
            multiple_sims=True, normalized=[amplitude_baseline,amplitude_baseline_hc], colorList=['C1','maroon']
        )  

In [ ]:
plot_fourier_analysis(
            xf_pos, [amplitude,amplitude_hc], [phase,phase_hc], freq,
            save_path=os.path.join(output_folder, f"fourier_{bin}bin_{sim_id}.png"),
            multiple_sims=True, normalized=True, colorList=['C1','maroon']
        )  

### Raster plot with population firing rate

In [ ]:
multiple_psth_plot([spikeData_lc,spikeData_hc], "", [cell_to_rank_lc,cell_to_rank_hc], [simulationData_lc,simulationData_hc],['low correlation state','high correlation state'], bin_width=bin, save_histogram=os.path.join(output_folder, f"psth_{sim_id}_{bin}bin_smoothed10.png"), figsize=(6,3),colorList = ['C1','maroon'])


In [ ]:
psth_plot(spikeData_hc, "all cells", cell_to_rank_hc, simulationData_hc, bin_width=bin, save_histogram=os.path.join(output_folder, f"psth_{sim_id}_{bin}bin_smoothed10.png"), smoothed=True,figsize=(7,7))

### Plot Fourier analysis:
1. Fourier spectrum
2. Power spectrum

In [ ]:

plot_fourier_analysis(
            xf_pos, amplitude, phase, freq,
            save_path=os.path.join(output_folder, f"fourier_{bin}bin_{sim_id}.png")
        )   

In [ ]:
output_folder

### Examples of use - sandbox

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.rcParams.update({'font.size': 18})

In [ ]:
baseline = simulations[simulations['sim_id']=='baseline']

In [ ]:
def getSpiking(data):

    config_path = data['source_file'].values[0]

    simulationData, allData, cellData, spikeData = load_data(config_path)

    return spikeData

In [ ]:
def getBaselineData(data, targetFreq, cellType=None):

    config_path = data['source_file'].values[0]

    simulationData, allData, cellData, spikeData = load_data(config_path)
    cell_to_rank = geom_rank_cell_nodes(cellData, simulationData)

    if cellType is not None:
        cell_types = extract_main_cell_types(spikeData, cellData, cell_to_rank)
        spikeData, cell_to_rank = cell_types[celltype]

    hist, _ = get_firingrate_hist(
                simulationData, spikeData, bin_width=5, n_neurons=cell_to_rank.shape[0], delay=1500 , time_stop=2500, smoothed=True)
    
    xf_pos, amplitude, yf = compute_fourier(hist, 5 * 1e-3, referenced=True, sim_id=str(targetFreq)+'Hz1Vm')

    return xf_pos, amplitude, yf

In [ ]:
def getBaselineComplex(data, targetFreq,cellType=None):

    xf_pos, amplitude, yf = getBaselineData(data, targetFreq,cellType)

    return yf[np.argmin(np.abs(xf_pos-targetFreq))]

In [ ]:
def getBaselinePower(data, targetFreq,cellType=None):

    xf_pos, amplitude, yf = getBaselineData(data, targetFreq,cellType)

    normalized_amp = amplitude #/ amplitude[0]

    power = normalized_amp ** 2
    power_norm = power / np.sum(power)

    return power_norm[np.argmin(np.abs(xf_pos-targetFreq))]

In [ ]:
def _windowed_spike_phases(
    spike_data,
    t_stop,
    freq,
    phase0=0.0,
    phase_orientation=0.0,
    stim_start=0.0,
    delay=0.0,
) -> np.ndarray:
    spike_times_s = np.array(spike_data.index.tolist()) * 1e-3

    print(spike_times_s)
    
    spike_times_s = spike_times_s[spike_times_s > delay]

    print(spike_times_s)
    
    spike_times_s = spike_times_s[spike_times_s < t_stop]

    print(spike_times_s)
    
    return 2 * np.pi * freq * (spike_times_s - stim_start) + phase0 + phase_orientation

def getBaselinePLV(data, targetFreq,cellType=None):

    spikeData = getSpiking(data)

    theta = _windowed_spike_phases(
        spikeData,
        2500*1e-3,
        targetFreq,
        delay=1500*1e-3,
        stim_start=1500*1e-3
    )
    if len(theta) == 0:
        return np.nan
    return np.abs(np.mean(np.exp(1j * theta)))

In [ ]:
stimFreqs = [7,10,15]

plt.figure(figsize=(8,8))

for j, freq in enumerate(stimFreqs):
    data = simulations[simulations['stim_freq']==freq]

    goodIdx = []
    for i in range(len(data)):
        if 'highCorrelation' not in data.iloc[i]['sim_id'] and 'delay' not in data.iloc[i]['sim_id']:
            goodIdx.append(i)
    
    data = data.iloc[goodIdx]

    amplitudes = data['stim_amp']
    power = data['amplitude_norm']

    # amplitudes = pd.concat([pd.Series(0),amplitudes],ignore_index=True)
    # power = pd.concat([pd.Series(getBaselinePower(baseline,freq)),power],ignore_index=True)

    plt.plot(amplitudes,power,'-',label=f'{freq} Hz')
    plt.scatter(amplitudes,power,s=amplitudes*20,c='C'+str(j))

plt.xlabel('Stimulus amplitude (V/m)')
plt.ylabel('Normalized power at target frequency')
plt.legend()
plt.tight_layout()

In [ ]:
stimFreqs = [7,10,15]

plt.figure(figsize=(8,8))

for j, freq in enumerate(stimFreqs):
    data = simulations[simulations['stim_freq']==freq]

    goodIdx = []
    for i in range(len(data)):
        if 'highCorrelation' not in data.iloc[i]['sim_id'] and 'delay' not in data.iloc[i]['sim_id']:
            goodIdx.append(i)
    
    data = data.iloc[goodIdx]

    amplitudes = data['stim_amp']
    power = data['power_norm']

    amplitudes = pd.concat([pd.Series(0),amplitudes],ignore_index=True)
    power = pd.concat([pd.Series(getBaselinePower(baseline,freq)),power],ignore_index=True)

    plt.plot(amplitudes,power,'-',label=f'{freq} Hz')
    plt.scatter(amplitudes,power,s=amplitudes*20,c='C'+str(j))

plt.xlabel('Stimulus amplitude (V/m)')
plt.ylabel('Normalized power at target frequency')
plt.legend()
plt.tight_layout()

plt.savefig('output/power.png',dpi=600)

In [ ]:
amplitudes

In [ ]:
stimFreqs = [7,10,15]

allDataReal = np.real(simulations['complex'].astype(complex))
allDataImag = np.imag(simulations['complex'].astype(complex))

maxReal = np.max(np.abs(allDataReal))
maxImag = np.max(np.abs(allDataImag))

plotLim = 8

plt.figure(figsize=(8,8))
plt.grid(True)
plt.axhline(0, color='black', linewidth=0.8)
plt.axvline(0, color='black', linewidth=0.8)

for j, freq in enumerate(stimFreqs):
    data = simulations[simulations['stim_freq']==freq]

    goodIdx = []
    for i in range(len(data)):
        if 'highCorrelation' not in data.iloc[i]['sim_id'] and 'delay' not in data.iloc[i]['sim_id']:
            goodIdx.append(i)
    
    data = data.iloc[goodIdx]

    amplitudes = data['stim_amp']

    realParts = np.real(data['complex'].astype(complex))
    imagParts = np.imag(data['complex'].astype(complex))

    baselineComplex = getBaselineComplex(baseline,freq)

    realParts = np.insert(realParts,0,np.real(baselineComplex))
    imagParts = np.insert(imagParts,0,np.imag(baselineComplex))
    amplitudes = pd.concat([pd.Series(0),amplitudes],ignore_index=True)

    plt.plot(realParts,imagParts,'-',c='C'+str(j),label=f'{freq} Hz')
    plt.scatter(realParts,imagParts,s=(amplitudes+.01)*20,c='C'+str(j))

plt.xlim([-2,plotLim])
plt.ylim([-plotLim,2])

data = simulations[simulations['stim_freq']==7]
amplitudes = data['stim_amp']
amplitudes = np.insert(amplitudes,0,0)
freq_legend = plt.legend()


size_handles = [
    plt.scatter([], [], s=(a+.01) * 20, color='grey')
    for a in amplitudes
]

size_legend = plt.legend(
    size_handles,
    [f'{a}' for a in amplitudes],
    title='Stimulus amplitude (V/m)',
    loc='lower left'
)

plt.gca().add_artist(size_legend)

plt.xlabel('Real part')
plt.ylabel('Imaginary part')
plt.legend()
plt.tight_layout()
plt.savefig('output/complex.png',dpi=600)

In [ ]:
stimFreqs = [10]
delays = [None,25,50]
markers = ['-',':','--']

allDataReal = np.real(simulations['complex'].astype(complex))
allDataImag = np.imag(simulations['complex'].astype(complex))

maxReal = np.max(np.abs(allDataReal))
maxImag = np.max(np.abs(allDataImag))

plotLim = 8

plt.figure(figsize=(8,8))
plt.grid(True)
plt.axhline(0, color='black', linewidth=0.8)
plt.axvline(0, color='black', linewidth=0.8)

for j, freq in enumerate(stimFreqs):
    alldata = simulations[simulations['stim_freq']==freq]

    for k, delay in enumerate(delays):

        goodIdx = []
        for i in range(len(alldata)):
            if 'highCorrelation' not in alldata.iloc[i]['sim_id']:
                if delay is None and 'delay' not in alldata.iloc[i]['sim_id']:
                    goodIdx.append(i)
                elif str(delay) in alldata.iloc[i]['sim_id']:
                    goodIdx.append(i)
        
        data = alldata.iloc[goodIdx]
    
        amplitudes = data['stim_amp']
    
        realParts = np.real(data['complex'].astype(complex))
        imagParts = np.imag(data['complex'].astype(complex))
    
        baselineComplex = getBaselineComplex(baseline,freq)
    
        realParts = np.insert(realParts,0,np.real(baselineComplex))
        imagParts = np.insert(imagParts,0,np.imag(baselineComplex))
        amplitudes = pd.concat([pd.Series(0),amplitudes],ignore_index=True)
    
        plt.plot(realParts,imagParts,markers[k],c='C'+str(j),label=f'{freq} Hz')
        plt.scatter(realParts,imagParts,s=(amplitudes+.01)*20,c='C'+str(j))

# plt.xlim([-2,plotLim])
# plt.ylim([-plotLim,2])

# data = simulations[simulations['stim_freq']==7]
# amplitudes = data['stim_amp']
# amplitudes = np.insert(amplitudes,0,0)
# freq_legend = plt.legend()


# size_handles = [
#     plt.scatter([], [], s=(a+.01) * 20, color='grey')
#     for a in amplitudes
# ]

# size_legend = plt.legend(
#     size_handles,
#     [f'{a}' for a in amplitudes],
#     title='Stimulus amplitude (V/m)',
#     loc='lower left'
# )

# plt.gca().add_artist(size_legend)

plt.xlabel('Real part')
plt.ylabel('Imaginary part')
plt.legend()
plt.tight_layout()

In [ ]:
delay = 1550
true_delay = None
t0 = delay - true_delay if true_delay is not None else 0.0
print(t0)

In [ ]:
k

In [ ]:
stimFreqs = [7, 10, 15]

plt.figure(figsize=(8, 8))

allAmplitudes = []
allAngles = []

for j, freq in enumerate(stimFreqs):
    data = simulations[simulations['stim_freq'] == freq]

    goodIdx = []
    for i in range(len(data)):
        if 'highCorrelation' not in data.iloc[i]['sim_id']:
            goodIdx.append(i)

    data = data.iloc[goodIdx]

    complexVals = data['complex'].astype(complex)
    amplitudes = data['stim_amp']

    baselineComplex = getBaselineComplex(baseline, freq)

    complexVals = np.insert(complexVals, 0, baselineComplex)
    amplitudes = pd.concat([pd.Series([0]), amplitudes], ignore_index=True)

    angles = np.angle(complexVals) * 180 / np.pi

    allAngles.append(np.array(angles))
    allAmplitudes.append(np.array(amplitudes))

singleCellPhase = -34.79

# --- Build grouped bar plot ---
all_amp_values = np.sort(np.unique(np.concatenate(allAmplitudes)))
group_spacing = 1.8
x = np.arange(len(all_amp_values)) * group_spacing

bar_width = 0.25

for j, freq in enumerate(stimFreqs):
    # Map amplitudes to angles for this frequency
    amp_to_angle = dict(zip(allAmplitudes[j], allAngles[j]))

    # Fill missing amplitudes with NaN
    y = [amp_to_angle.get(a, np.nan) for a in all_amp_values]

    plt.bar(
        x + (j - (len(stimFreqs)-1)/2) * bar_width,
        y,
        width=bar_width,
        label=f'{freq} Hz Stimulation'
    )

# Horizontal reference line
plt.axhline(
    singleCellPhase,
    color='k',
    linestyle='--',
    linewidth=2,
    label='Single-cell phase delay'
)

plt.xticks(x, all_amp_values)
plt.xlabel('Stimulus Amplitude (V/m)')
plt.ylabel('Phase Delay (deg)')
plt.legend()

plt.tight_layout()
plt.savefig('output/angles.png', dpi=600)
plt.show()

In [ ]:
stimFreqs = [7, 10, 15]

plt.figure(figsize=(8, 8))

allAmplitudes = []
allAngles = []

for j, freq in enumerate(stimFreqs):
    data = simulations[simulations['stim_freq'] == freq]

    goodIdx = []
    for i in range(len(data)):
        if 'highCorrelation' not in data.iloc[i]['sim_id']:
            goodIdx.append(i)

    data = data.iloc[goodIdx]

    complexVals = data['complex'].astype(complex)
    amplitudes = data['stim_amp']

    baselineComplex = getBaselineComplex(baseline, freq)

    complexVals = np.insert(complexVals, 0, baselineComplex)
    amplitudes = pd.concat([pd.Series([0]), amplitudes], ignore_index=True)

    angles = np.angle(complexVals) * 180 / np.pi

    allAngles.append(np.array(angles))
    allAmplitudes.append(np.array(amplitudes))

# allAngles = np.array(allAngles)
# allAmplitudes = np.array(allAmplitudes)
singleCellPhase = -34.79

# --- Build grouped bar plot ---


bar_width = 0.25

for i in range(len(stimFreqs)):
    plt.plot(allAmplitudes[i],allAngles[i],c='C'+str(i),label=f'{stimFreqs[i]} Hz Stimulation')
    plt.scatter(allAmplitudes[i],allAngles[i],s=(allAmplitudes[i]+.01)*20,c='C'+str(i))

# Horizontal reference line
plt.axhline(
    singleCellPhase,
    color='k',
    linestyle='--',
    linewidth=2,
    label='Single-cell phase delay'
)

plt.xlabel('Stimulus Amplitude (V/m)')
plt.ylabel('Phase Delay (deg)')
plt.legend()

plt.tight_layout()
plt.savefig('output/angles.png', dpi=600)
plt.show()

In [ ]:
stimFreqs = [7, 10, 15]

plt.figure(figsize=(8, 8))

allAmplitudes = []
allAngles = []

for j, freq in enumerate(stimFreqs):
    data = simulations[simulations['stim_freq'] == freq]

    goodIdx = []
    for i in range(len(data)):
        if 'highCorrelation' not in data.iloc[i]['sim_id']:
            goodIdx.append(i)

    data = data.iloc[goodIdx]

    complexVals = data['complex'].astype(complex)
    amplitudes = data['stim_amp']

    baselineComplex = getBaselineComplex(baseline, freq)

    complexVals = np.insert(complexVals, 0, baselineComplex)
    amplitudes = pd.concat([pd.Series([0]), amplitudes], ignore_index=True)

    angles = np.angle(complexVals) * 180 / np.pi

    allAngles.append(np.array(angles))
    allAmplitudes.append(np.array(amplitudes))

# allAngles = np.array(allAngles)
# allAmplitudes = np.array(allAmplitudes)
singleCellPhases = [-31,-36,-44]

# --- Build grouped bar plot ---


bar_width = 0.25

for i in range(len(stimFreqs)):
    plt.plot(allAmplitudes[i],allAngles[i]-singleCellPhases[i],c='C'+str(i),label=f'{stimFreqs[i]} Hz Stimulation')
    plt.scatter(allAmplitudes[i],allAngles[i]-singleCellPhases[i],s=(allAmplitudes[i]+.01)*20,c='C'+str(i))

plt.xlabel('Stimulus Amplitude (V/m)')
plt.ylabel('Difference between column phase delay\nand single-neuron phase delay (degrees)')
plt.legend()

plt.tight_layout()
plt.savefig('output/angles.png', dpi=600)
plt.show()

In [ ]:
allAngles

In [ ]:
stimFreqs = [7,10,15]

plt.figure(figsize=(8,8))

for j, freq in enumerate(stimFreqs):
    data = simulations[simulations['stim_freq']==freq]

    goodIdx = []
    for i in range(len(data)):
        if 'highCorrelation' not in data.iloc[i]['sim_id']:
            goodIdx.append(i)
    
    data = data.iloc[goodIdx]

    amplitudes = data['stim_amp']
    plv = data['plv']

    amplitudes = pd.concat([pd.Series(0),amplitudes],ignore_index=True)
    plv = pd.concat([pd.Series(getBaselinePLV(baseline,freq)),plv],ignore_index=True)

    plt.plot(amplitudes,plv,'-',label=f'{freq} Hz')
    plt.scatter(amplitudes,plv,s=amplitudes*20,c='C'+str(j))

plt.xlabel('Stimulus amplitude (V/m)')
plt.ylabel('PLV')
plt.legend()
plt.tight_layout()
plt.savefig('output/plv.png',dpi=600)

In [ ]:
stimFreqs = [7,10,15]

plt.figure()

for j, freq in enumerate(stimFreqs):
    data = simulations[simulations['stim_freq']==freq]

    goodIdx = []
    for i in range(len(data)):
        if 'highCorrelation' not in data.iloc[i]['sim_id']:
            goodIdx.append(i)
    
    data = data.iloc[goodIdx]

    amplitudes = data['stim_amp']
    dfr = data['deltafr']
    power = data['power_norm']

    plt.plot(amplitudes,dfr,'-',label=f'{freq} Hz')
    plt.scatter(amplitudes,dfr,s=amplitudes*20,c='C'+str(j))

plt.xlabel('Stimulus amplitude (V/m)')
plt.ylabel(r'$\Delta$ FR')
plt.legend()
plt.tight_layout()
plt.savefig('output/dfr.png',dpi=600)

In [ ]:
cellTypes = ['PYR','INT_EXC','INT_INH']
colors = ['r','brown','cyan']
markers = ['^','s','X']

plt.figure()

stimFreq = 10

data = simulations[simulations['stim_freq']==stimFreq]

goodIdx = []
for i in range(len(data)):
    if 'highCorrelation' not in data.iloc[i]['sim_id']:
        goodIdx.append(i)

data = data.iloc[goodIdx]

amplitudes = data['stim_amp']

power = data['power_norm']*data['total_power']

# amplitudes = pd.concat([pd.Series(0),amplitudes],ignore_index=True)
# power = pd.concat([pd.Series(getBaselinePower(baseline,stimFreq)),power],ignore_index=True)

plt.plot(amplitudes,power,'-',color='C1',label=f'All cells')
plt.scatter(amplitudes,power,s=amplitudes*20,c='C1',)

for j, celltype in enumerate(cellTypes):
    
    
    power = data['total_power_'+celltype]
    # power = pd.concat([pd.Series(getBaselinePower(baseline,stimFreq,celltype)),power],ignore_index=True)

    plt.plot(amplitudes,power,'-',color=colors[j],label=f'{celltype}')
    plt.scatter(amplitudes,power,s=amplitudes*20,c=colors[j],marker=markers[j])

plt.xlabel('Stimulus amplitude (V/m)')
plt.ylabel('Normalized power at target frequency')
plt.legend()
plt.tight_layout()
# plt.savefig('output/power_by_type.png',dpi=600)

In [ ]:
cellTypes = ['PYR','INT_EXC','INT_INH']
colors = ['r','brown','cyan']
markers = ['^','s','X']

plt.figure()

stimFreq = 10

data = simulations[simulations['stim_freq']==stimFreq]

goodIdx = []
for i in range(len(data)):
    if 'highCorrelation' not in data.iloc[i]['sim_id']:
        goodIdx.append(i)

data = data.iloc[goodIdx]

amplitudes = data['stim_amp']

power = data['power_norm']

amplitudes = pd.concat([pd.Series(0),amplitudes],ignore_index=True)
power = pd.concat([pd.Series(getBaselinePower(baseline,stimFreq)),power],ignore_index=True)

plt.plot(amplitudes,power,'-',color='C1',label=f'All cells')
plt.scatter(amplitudes,power,s=amplitudes*20,c='C1',)

for j, celltype in enumerate(cellTypes):
    
    
    power = data['power_norm_'+celltype]
    power = pd.concat([pd.Series(getBaselinePower(baseline,stimFreq,celltype)),power],ignore_index=True)

    plt.plot(amplitudes,power,'-',color=colors[j],label=f'{celltype}')
    plt.scatter(amplitudes,power,s=amplitudes*20,c=colors[j],marker=markers[j])

plt.xlabel('Stimulus amplitude (V/m)')
plt.ylabel('Normalized power at target frequency')
plt.legend()
plt.tight_layout()


In [ ]:
cellTypes = ['INT_EXC','INT_INH','PYR']
colors = ['brown','cyan','r',]
markers = ['s','X','^']

stimFreq = 10

plt.figure(figsize=(8,8))
plt.grid(True)
plt.axhline(0, color='black', linewidth=0.8)
plt.axvline(0, color='black', linewidth=0.8)

allDataReal = np.real(simulations['complex'].astype(complex))
allDataImag = np.imag(simulations['complex'].astype(complex))

plotLim = 8

data = simulations[simulations['stim_freq']==stimFreq]

goodIdx = []
for i in range(len(data)):
    if 'highCorrelation' not in data.iloc[i]['sim_id']:
        goodIdx.append(i)

data = data.iloc[goodIdx]

for j, celltype in enumerate(cellTypes):
    
    complexvals = data['complex_'+celltype]

    realParts = np.real(complexvals.astype(complex))
    imagParts = np.imag(complexvals.astype(complex))

    realParts = np.insert(realParts,0,np.real(getBaselineComplex(baseline,stimFreq,celltype)))
    imagParts = np.insert(imagParts,0,np.imag(getBaselineComplex(baseline,stimFreq,celltype)))

    plt.plot(realParts,imagParts,'-',color=colors[j],label=f'{celltype}')
    plt.scatter(realParts,imagParts,s=(amplitudes+.01)*20,c=colors[j],marker=markers[j])


amplitudes = data['stim_amp']
amplitudes = pd.concat([pd.Series(0),amplitudes],ignore_index=True)

realParts = np.real(data['complex'].astype(complex))
imagParts = np.imag(data['complex'].astype(complex))

realParts = np.insert(realParts,0,np.real(getBaselineComplex(baseline,stimFreq)))
imagParts = np.insert(imagParts,0,np.imag(getBaselineComplex(baseline,stimFreq)))

plt.plot(realParts,imagParts,'-',c='C1',label=f'All cells')
plt.scatter(realParts,imagParts,s=amplitudes*20,c='C1')

from matplotlib.lines import Line2D

celltype_handles = [
    Line2D(
        [0], [0],
        color=colors[j],
        marker=markers[j],
        linestyle='-',
        markersize=8,
        label=cellTypes[j]
    )
    for j in range(len(cellTypes))
]

allcells_handle = Line2D(
    [0], [0],
    color='C1',
    linestyle='-',
    marker='o',
    markersize=8,
    label='All cells'
)

freq_legend = plt.legend(
    handles= celltype_handles + [allcells_handle],
    loc='upper right',
    ncol=2
)

size_handles = [
    plt.scatter([], [], s=(a+.01) * 20, color='grey')
    for a in amplitudes
]

size_legend = plt.legend(
    size_handles,
    [f'{a}' for a in amplitudes],
    title='Stimulus amplitude (V/m)',
    loc='lower left',
    ncol=2
)

plt.gca().add_artist(size_legend)
plt.gca().add_artist(freq_legend)

plt.xlabel('Real part')
plt.ylabel('Imaginary part')
plt.xlim([-4,8])
plt.ylim([-8,4])
plt.tight_layout()
plt.title('10 Hz stimulus')
plt.savefig('output/complex_by_type.png',dpi=600)


In [ ]:
np.angle(data['complex'].astype(complex).iloc[-1])*180/np.pi

In [ ]:
stimFreqs = [7,10,15]

amplitudes = np.empty((3,4),dtype=complex)
firsthalves = np.empty((3,4),dtype=complex)
secondhalves = np.empty((3,4),dtype=complex)

plt.figure()

for j, freq in enumerate(stimFreqs):
    
    data = simulations[simulations['stim_freq']==freq]

    goodIdx = []
    for i in range(len(data)):
        if 'highCorrelation' not in data.iloc[i]['sim_id']:
            goodIdx.append(i)
    
    data = data.iloc[goodIdx]

    for i in range(len(data)):

        amplitudes[j,i] = data.iloc[i]['stim_amp']

        config_path = data.iloc[i]['source_file']

        simulationData, allData, cellData, spikeData = load_data(config_path)
        cell_to_rank = geom_rank_cell_nodes(cellData, simulationData)

        hist_firsthalf, _ = get_firingrate_hist(
                    simulationData, spikeData, bin_width=5, n_neurons=cell_to_rank.shape[0], delay=1500 , time_stop=2000, smoothed=True)
        
        xf_pos_1, _, yf_1 = compute_fourier(hist_firsthalf, 5 * 1e-3, referenced=True, sim_id=data.iloc[i]['sim_id'])

        target_1 = yf_1[np.argmin(np.abs(xf_pos_1-freq))]

        firsthalves[j,i] = target_1

        hist_secondhalf, _ = get_firingrate_hist(
                    simulationData, spikeData, bin_width=5, n_neurons=cell_to_rank.shape[0], delay=2000 , time_stop=2500, smoothed=True)
        
        xf_pos_2, _, yf_2 = compute_fourier(hist_secondhalf, 5 * 1e-3, referenced=True, sim_id=data.iloc[i]['sim_id'])

        target_2 = yf_2[np.argmin(np.abs(xf_pos_2-freq))]

        secondhalves[j,i] = target_2



In [ ]:
amplitudes[j]

In [ ]:
amplitudes = np.abs(amplitudes)

plt.figure(figsize=(6,3))

plt.subplot(1,2,1)
plt.grid(True)
plt.axhline(0, color='black', linewidth=0.8)
plt.axvline(0, color='black', linewidth=0.8)
plt.xlim([-5,5])
plt.ylim([-5,5])
for j, freq in enumerate(stimFreqs):

    plt.plot(np.real(firsthalves[j]),np.imag(firsthalves[j]),'-',c='C'+str(j),label=f'{freq} Hz')
    plt.scatter(np.real(firsthalves[j]),np.imag(firsthalves[j]),s=amplitudes[j]*20,c='C'+str(j))

plt.subplot(1,2,2)
plt.grid(True)
plt.axhline(0, color='black', linewidth=0.8)
plt.axvline(0, color='black', linewidth=0.8)
plt.xlim([-5,5])
plt.ylim([-5,5])
for j, freq in enumerate(stimFreqs):

    plt.plot(np.real(secondhalves[j]),np.imag(secondhalves[j]),'-',c='C'+str(j),label=f'{freq} Hz')
    plt.scatter(np.real(secondhalves[j]),np.imag(secondhalves[j]),s=amplitudes[j]*20,c='C'+str(j))

In [ ]:
firsthalves

In [ ]:
# I want to look at all the plv values for all simulations with 15Hz target frequency


data_15 = simulations[simulations['stim_freq'] == 15]
plv_values = data_15['plv'].values
amplitude = data_15['stim_amp'].values

#Say you want to add the baseline value -> you know this is simulation sim_id = resting

#----------------------------------------
# please be careful because different machines for deployement may have different baseline values
# make sure they correspond 
# ---------------------------------------

#plv value for resting state is different based on the target frerquency of the stimulation, so make sure to use the correct one

#for 15HZ
freq = data_15['stim_freq'].values[0]
phase0 = data_15['stim_phase'].values[0]
phase_orientation = data_15['stim_orientation'].values[0]

simulationData, allData, cellData, spikeData = load_data(config_path)
cell_to_rank = geom_rank_cell_nodes(cellData, simulationData)
delay = 500 #take an initial delay of 500ms to make sure we are in the steady state of the stimulation, and not in the transient part
time_stop = simulations[simulations['sim_id'] == 'resting']['tstop'].values[0] #take the same time stop as the resting state simulation to make sure we are comparing the same time window for the plv calculation
baseline_plv15 = get_plv(
    spikeData,
    time_stop * 1e-3,
    freq,
    phase0=phase0,
    phase_orientation=phase_orientation,
    delay=delay * 1e-3,
)

plv_values = np.append(plv_values, baseline_plv15)  
amplitude = np.append(amplitude, 0)

plt.plot(amplitude, plv_values, 'o')
plt.xlabel('Amplitude')
plt.ylabel('PLV')
plt.title('PLV vs Amplitude for 15Hz stimulation')
plt.show()

# I want to do this now only for the pyramidal cells, now

cell_types_dict = extract_main_cell_types(spikeData, cellData, cell_to_rank)
spikeData_pyr, cell_to_rank_pyr = cell_types_dict['PYR']

plv_values_15_pyramidal = data_15['plv_PYR'].values
baseline_plv15_PYR = get_plv(
    spikeData_pyr,
    time_stop * 1e-3,
    freq,
    phase0=phase0,
    phase_orientation=phase_orientation,
    delay=delay * 1e-3,
)
plv_values_15_pyramidal = np.append(plv_values_15_pyramidal, baseline_plv15_PYR)
plt.plot(amplitude, plv_values_15_pyramidal, 'o')
plt.xlabel('Amplitude')
plt.ylabel('PLV (Pyramidal cells)')
plt.title('PLV vs Amplitude for 15Hz stimulation (Pyramidal cells)')
plt.show()


In [ ]:
singleCell = pd.read_csv('/mnt/c/Users/tharayil/Downloads/singleCell_15.csv')

f = 15

time = singleCell.iloc[:,0].values
soma = singleCell.iloc[:,1].values
tstart = np.argmin(np.abs(time-0.5))
soma = soma[tstart:]
time = time[tstart:]
time = time - np.min(time)

signal = np.sin(2*np.pi*f*time)

In [ ]:
time

In [ ]:
import numpy as np
from scipy.signal import correlate

# Sampling parameters
fs = len(time)/np.max(time)  # Hz

# Time vector

# Signals
x = soma
y = -signal 

# Cross-correlation
corr = correlate(y, x, mode='full')
lags = np.arange(-len(x) + 1, len(x))

# Delay estimate
lag_samples = lags[np.argmax(corr)]
delay = lag_samples / fs

# Phase difference
phase_rad = 2 * np.pi * f * delay
phase_deg = np.degrees(phase_rad)

# Wrap to [-180, 180]
phase_deg = (phase_deg + 180) % 360 - 180

print(f"Delay: {delay:.6f} s")
print(f"Phase lag: {phase_deg:.2f} degrees")

In [ ]:
(2*np.pi*f*(np.argmax(soma[:3000])-np.argmax(-signal[:3000]))/40000)*180/np.pi

In [ ]:
(31+36+44)/3

In [ ]:
plt.figure()
plt.subplot(211)
plt.plot(soma[:5000])
plt.subplot(212)
plt.plot(-signal[:5000])

In [ ]:
# FFTs
X = np.fft.rfft(soma)
Y = np.fft.rfft(signal)

fs = len(time)/np.max(time)

# Frequency bins
freqs = np.fft.rfftfreq(len(soma), d=1/fs)

# Index of frequency of interest
idx = np.argmin(np.abs(freqs - 10))

# Phase difference
phase_rad = np.angle(Y[idx]) - np.angle(X[idx])
phase_deg = np.degrees(phase_rad)

# Wrap to [-180, 180]
phase_deg = (phase_deg + 180) % 360 - 180

print(f"Phase lag: {phase_deg:.2f} degrees")